In [0]:
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("ChicagoEvictionsMerge") \
    .getOrCreate()


# 1. LOAD & CLEAN CHICAGO HEALTH ATLAS CSV

cha_path = "Chicago Health Atlas Data Download - Census Tracts.csv"
cha_raw = spark.read.option("header", "true").csv(cha_path)

# Drop row 0 (Spark DataFrames are unordered, so assign a row number based on file order)
w = Window.orderBy(F.monotonically_increasing_id())
cha = (
    cha_raw
    .withColumn("_row_num", F.row_number().over(w))
    .filter(F.col("_row_num") > 1)
    .drop("_row_num")
)

# Standardize GEOID as an 11-character zero-padded string
cha = cha.withColumn(
    "geoid",
    F.lpad(F.col("GEOID").cast("double").cast("long").cast("string"), 11, "0")
)

# Convert all indicator columns to numeric (non-numeric strings safely become null, matching errors="coerce")
indicator_cols = [c for c in cha.columns if c not in ["Layer", "Name", "GEOID", "geoid"]]
for col_name in indicator_cols:
    cha = cha.withColumn(col_name, F.col(col_name).cast("double"))

# Prefix Chicago Health Atlas columns to avoid collisions and drop unused columns
cha = (
    cha
    .withColumnRenamed("Name", "cha_tract_name")
    .withColumnsRenamed({col_name: f"cha_{col_name}" for col_name in indicator_cols})
    .drop("Layer", "GEOID")
)

print(f"Cleaned Chicago Health Atlas shape: ({cha.count()}, {len(cha.columns)})")


# 2. LOAD & FILTER EVICTION LAB (2000–2018)

el_matches = glob.glob("tract_proprietary_valid_2000_2018_y2024m12*")
if not el_matches:
    raise FileNotFoundError("Could not find tract_proprietary_valid_2000_2018_y2024m12 in current folder.")
el_path = el_matches[0]
print(f"Reading Eviction Lab file: {el_path}")

# Read CSV directly (Spark distributes file reading across partitions natively, no chunking needed)
el_raw = spark.read.option("header", "true").csv(el_path)

# Strip whitespace from column names
el_raw = el_raw.toDF(*[c.strip() for c in el_raw.columns])
sample_cols = el_raw.columns
print(f"Eviction Lab columns: {sample_cols}")

geoid_col = next((c for c in sample_cols if c.lower() in ["geoid", "fips"]), None)
if geoid_col is None:
    raise KeyError(f"Could not find a geoid column in: {sample_cols}")

# Set FIPS prefix:
#   "17031" = Cook County only
#   "17"    = Entire State of Illinois
FIPS_PREFIX = "17031"

eviction_lab = el_raw.withColumn(
    "geoid",
    F.lpad(F.regexp_replace(F.trim(F.col(geoid_col)), r"\.0$", ""), 11, "0")
)

if geoid_col != "geoid":
    eviction_lab = eviction_lab.drop(geoid_col)

eviction_lab = eviction_lab.filter(F.col("geoid").startswith(FIPS_PREFIX))

year_col = next((c for c in eviction_lab.columns if c.lower() == "year"), None)
if year_col:
    eviction_lab = eviction_lab.withColumn(year_col, F.col(year_col).cast("long"))

print(f"Filtered Eviction Lab ({FIPS_PREFIX}) shape: ({eviction_lab.count()}, {len(eviction_lab.columns)})")


# 3. MERGE BOTH DATASETS ON GEOID

# Broadcast `cha` since census tract metadata is small relative to multi-year panel data
chicago_panel = eviction_lab.join(F.broadcast(cha), on="geoid", how="inner")
cook_county_panel = eviction_lab.join(F.broadcast(cha), on="geoid", how="left")

if year_col:
    chicago_panel = chicago_panel.withColumn(
        "is_2018_cha_snapshot",
        (F.col(year_col) == 2018).cast("int")
    )

print(f"Merged Chicago panel shape:     ({chicago_panel.count()}, {len(chicago_panel.columns)})")
print(f"Merged Cook County panel shape: ({cook_county_panel.count()}, {len(cook_county_panel.columns)})")

# Save outputs
# Note: Spark writes CSVs as directories of part-files by default; coalesce(1) writes a single file inside the folder
chicago_panel.coalesce(1).write.mode("overwrite").option("header", "true").csv("chicago_evictions_combined_2000_2018")
cook_county_panel.coalesce(1).write.mode("overwrite").option("header", "true").csv("cook_county_evictions_combined_2000_2018")
print("Saved both combined CSVs successfully.")